# Fake.Br — Deep Learning (compatível com o repositório do grupo)

Mineração de Texto, 2ª VA — modelos **Word2Vec + BiLSTM** e **BERTimbau Base (fine-tuning)**.

**Leia antes de rodar:** o grupo já definiu uma divisão oficial em `data/split.csv`, com pares alinhados e textos `size_normalized_texts`. Este notebook **não** cria uma divisão nova. Ele lê `src.dados.carregar_corpus()` do repositório `LeanderHeitor/fakebr-classificacao`.

- Os rótulos são `0 = verdadeira` e `1 = falsa`.
- Mantenha `MODO_RAPIDO=True` para testes técnicos; **use `False` para gerar os resultados comparáveis do trabalho**.
- As previsões finais seguem o padrão do repositório: `resultados/predictions_bilstm.csv` e `resultados/predictions_bertimbau.csv` com as colunas **`id,y_true,y_pred,score`** (probabilidade de notícia falsa).
- Não subir modelos grandes nem tokens/segredos para o GitHub.


### 1. Dependências
As bibliotecas e os modelos são baixados **no ambiente temporário do Colab**, não no seu computador. Não é necessário configurar chave de API.

In [ ]:
%pip -q install "transformers>=4.56,<5" "accelerate>=1.1" "datasets>=3,<5" "gensim>=4.3.3" scikit-learn pandas matplotlib

### 2. Configurações e ambiente

Execute as células em ordem. A primeira execução do BERT requer GPU e download dos pesos no **ambiente remoto do Colab**. Não utilize resultados do modo rápido na tabela final do trabalho.

In [ ]:
import os, re, random, time, json, gc, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

MODO_RAPIDO = False      # True só para testes técnicos; False para avaliação final (7.200 notícias)
PARES_RAPIDOS = 250      # 250 pares = 500 notícias
REPO_DIR = Path("/content/fakebr-classificacao")  # repositório oficial do grupo

PASTA_RESULTADOS = Path('/content/resultados_fakebr_dl')
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| Dispositivo:', DEVICE)
if DEVICE.type == 'cpu':
    print('ATENÇÃO: para o BERTimbau, ative uma GPU em Ambiente de execução > Alterar tipo de ambiente.')

### 3. Carregar o **corpus oficial do grupo**

O módulo `src/dados.py` já define a versão de textos normalizados por tamanho e usa os IDs da divisão em `data/split.csv`. Isso evita treinar nos textos completos ou misturar os pares relacionados entre conjuntos.

In [ ]:
# Faz o clone NO COLAB (não baixa arquivos no notebook pessoal).
if not REPO_DIR.exists():
    subprocess.run(['git','clone','-q','https://github.com/LeanderHeitor/fakebr-classificacao.git',str(REPO_DIR)],check=True)
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0,str(REPO_DIR))
from src.dados import carregar_corpus, ARQ_SPLIT

oficial = carregar_corpus()
# Colunas do grupo: id, par, rotulo, categoria, split, texto.
assert set(['id','par','rotulo','split','texto']).issubset(oficial.columns)
df = oficial.rename(columns={'texto':'text','rotulo':'label','par':'group_id'}).copy()
assert df['id'].is_unique and df['label'].isin([0,1]).all()
assert df['split'].isin(['treino','validacao','teste']).all()
assert (df.groupby('group_id')['split'].nunique()==1).all(), 'Vazamento de pares entre conjuntos!'

if MODO_RAPIDO:
    # Subconjunto apenas para verificar se o código roda. Preserva a divisão oficial.
    pares=np.array(sorted(df.group_id.unique()))
    sorteados=np.random.default_rng(SEED).choice(pares,size=min(PARES_RAPIDOS,len(pares)),replace=False)
    df=df[df.group_id.isin(sorteados)].copy()

print(f'Divisão oficial: {ARQ_SPLIT} | Notícias: {len(df)} | Pares: {df.group_id.nunique()}')
print(pd.crosstab(df['split'], df['label']).rename(columns={0:'verdadeiras',1:'falsas'}))
display(df[['id','group_id','split','label','text']].head(2))

### 4. Usar treino, validação e teste **já definidos pelo grupo**

Ninguém sorteia novos conjuntos. Os três são filtrados da coluna `split` oficial.

In [ ]:
treino=df[df.split=='treino'].reset_index(drop=True)
validacao=df[df.split=='validacao'].reset_index(drop=True)
teste=df[df.split=='teste'].reset_index(drop=True)
assert all(not parte.empty for parte in [treino,validacao,teste]), 'Amostra pequena demais: faltou pelo menos um dos conjuntos.'
for a,b in [(treino,validacao),(treino,teste),(validacao,teste)]:
    assert not set(a.group_id)&set(b.group_id), 'Par alinhado em duas divisões!'
for nome,tabela in [('treino',treino),('validacao',validacao),('teste',teste)]:
    print(f'{nome}: {len(tabela)} notícias | rotulos={tabela.label.value_counts().to_dict()}')
    if not MODO_RAPIDO:
        # Salva IDs com os quais as previsões finais serão alinhadas.
        assert set(tabela.id)==set(oficial.loc[oficial.split==nome,'id'])
print('Modelos treinados e avaliados nos conjuntos de IDs oficiais.')

---
## Modelo A · Word2Vec + BiLSTM (PyTorch)
O Word2Vec é treinado **somente nos textos de treino**, evitando vazamento. A BiLSTM recebe sequências de vetores de palavras e aprende uma classificação binária. Para uma comparação justa, o BERTimbau também será avaliado sobre o mesmo conjunto de teste.

In [ ]:
from gensim.models import Word2Vec
from collections import Counter
from torch import nn
from torch.utils.data import Dataset, DataLoader

def tokenizar(texto):
    return re.findall(r"(?u)\b\w+\b", str(texto).lower())

tokens_treino=[tokenizar(txt) for txt in treino.text]
MAX_PALAVRAS_LSTM=256
DIM_EMBEDDING=100
MAX_VOCAB=30000

contagem=Counter(tok for documento in tokens_treino for tok in documento)
vocab=['<PAD>','<UNK>']+[t for t,n in contagem.most_common(MAX_VOCAB-2) if n>=2]
word2idx={word:i for i,word in enumerate(vocab)}
print('Tamanho do vocabulário:',len(vocab))

w2v=Word2Vec(sentences=tokens_treino, vector_size=DIM_EMBEDDING, window=5,
             min_count=2, workers=2, sg=1, epochs=8, seed=SEED)
matriz=np.random.default_rng(SEED).normal(0,0.05,(len(vocab),DIM_EMBEDDING)).astype('float32')
matriz[0]=0.0
for palavra,i in word2idx.items():
    if palavra in w2v.wv:
        matriz[i]=w2v.wv[palavra]

class DatasetNoticias(Dataset):
    def __init__(self,tabela):
        self.y=tabela.label.to_numpy(dtype='int64')
        self.seqs=[]
        for texto in tabela.text:
            ids=[word2idx.get(w,1) for w in tokenizar(texto)[:MAX_PALAVRAS_LSTM]]
            self.seqs.append(ids or [1])
    def __len__(self):return len(self.y)
    def __getitem__(self,i):return self.seqs[i],int(self.y[i])

def collate(batch):
    seqs, ys=zip(*batch)
    comprimentos=torch.tensor([len(s) for s in seqs],dtype=torch.long)
    max_len=int(comprimentos.max())
    X=torch.zeros(len(seqs),max_len,dtype=torch.long)
    for i,seq in enumerate(seqs): X[i,:len(seq)]=torch.tensor(seq)
    return X,comprimentos,torch.tensor(ys,dtype=torch.long)

loader_treino=DataLoader(DatasetNoticias(treino),batch_size=64,shuffle=True,collate_fn=collate)
loader_val=DataLoader(DatasetNoticias(validacao),batch_size=128,shuffle=False,collate_fn=collate)
loader_teste=DataLoader(DatasetNoticias(teste),batch_size=128,shuffle=False,collate_fn=collate)

In [ ]:
class ClassificadorBiLSTM(nn.Module):
    def __init__(self, pesos_embeddings):
        super().__init__()
        tensor=torch.tensor(pesos_embeddings,dtype=torch.float32)
        self.embedding=nn.Embedding.from_pretrained(tensor,freeze=False,padding_idx=0)
        self.lstm=nn.LSTM(input_size=tensor.shape[1],hidden_size=64,
                          batch_first=True,bidirectional=True)
        self.dropout=nn.Dropout(0.35)
        self.classificador=nn.Linear(128,2)
    def forward(self, x, comprimentos):
        embeddings=self.embedding(x)
        packed=nn.utils.rnn.pack_padded_sequence(embeddings,comprimentos.cpu(),
                                                 batch_first=True,enforce_sorted=False)
        _,(h_n,_) = self.lstm(packed)
        representacao=torch.cat((h_n[-2],h_n[-1]),dim=1)
        return self.classificador(self.dropout(representacao))

bilstm=ClassificadorBiLSTM(matriz).to(DEVICE)
print('Parâmetros treináveis:',sum(p.numel() for p in bilstm.parameters() if p.requires_grad))

In [ ]:
def prever_lstm(modelo,loader):
    modelo.eval(); saidas=[]; alvos=[]
    with torch.no_grad():
        for X,comprimentos,y in loader:
            logits=modelo(X.to(DEVICE),comprimentos)
            saidas.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            alvos.extend(y.numpy().tolist())
    return np.array(alvos),np.array(saidas)

opt=torch.optim.AdamW(bilstm.parameters(),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()
EPOCAS_LSTM=3 if MODO_RAPIDO else 8
historico_lstm=[]; melhor_f1=-1; melhor_estado=None; paciencia=2; sem_melhoria=0
for epoca in range(1,EPOCAS_LSTM+1):
    inicio=time.time(); bilstm.train(); soma=0
    for X,comprimentos,y in loader_treino:
        opt.zero_grad()
        logits=bilstm(X.to(DEVICE),comprimentos)
        perda=loss_fn(logits,y.to(DEVICE))
        perda.backward()
        nn.utils.clip_grad_norm_(bilstm.parameters(),max_norm=1.0)
        opt.step(); soma += perda.item()*len(y)
    y_val,p_val=prever_lstm(bilstm,loader_val)
    f1=f1_score(y_val,p_val,average='macro',zero_division=0)
    historico_lstm.append({'epoca':epoca,'loss_treino':soma/len(treino),'f1_macro_val':f1})
    print(f'Época {epoca}: loss={soma/len(treino):.4f} | F1-macro val={f1:.4f} | tempo={time.time()-inicio:.1f}s')
    if f1>melhor_f1:
        melhor_f1=f1
        melhor_estado={k:v.cpu().clone() for k,v in bilstm.state_dict().items()}
        sem_melhoria=0
    else:
        sem_melhoria+=1
        if sem_melhoria>=paciencia:
            print('Early stopping.');break
bilstm.load_state_dict(melhor_estado)
pd.DataFrame(historico_lstm).to_csv(PASTA_RESULTADOS/'historico_bilstm.csv',index=False)

In [ ]:
y_lstm,p_lstm=prever_lstm(bilstm,loader_teste)
print('BiLSTM — TESTE (não foi usado para escolher épocas):')
print(classification_report(y_lstm,p_lstm,target_names=['Verdadeira','Falsa'],digits=3,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_lstm,p_lstm,display_labels=['Verdadeira','Falsa'],cmap='Blues')
plt.title('Word2Vec + BiLSTM — teste');plt.show()

def calcular_metricas(y,p,modelo):
    return {'modelo':modelo,'accuracy':accuracy_score(y,p),
            'precision_fake':precision_score(y,p,pos_label=1,zero_division=0),
            'recall_fake':recall_score(y,p,pos_label=1,zero_division=0),
            'f1_fake':f1_score(y,p,pos_label=1,zero_division=0),
            'f1_macro':f1_score(y,p,average='macro',zero_division=0)}

metricas=[calcular_metricas(y_lstm,p_lstm,'Word2Vec + BiLSTM')]
previsoes=teste[['text','label','group_id']].copy()
previsoes['pred_bilstm']=p_lstm
print(pd.DataFrame(metricas).round(4).to_string(index=False))
# Exportação padronizada — usa a ordem de loader_teste (shuffle=False) e os IDs de teste.
def probabilidades_falsa_bilstm(modelo, loader):
    modelo.eval()
    partes=[]
    with torch.no_grad():
        for X, comprimentos, _ in loader:
            logits = modelo(X.to(DEVICE), comprimentos)
            partes.extend(torch.softmax(logits, dim=1)[:,1].cpu().numpy().tolist())
    return np.asarray(partes)

scores_bilstm=probabilidades_falsa_bilstm(bilstm,loader_teste)
assert len(scores_bilstm)==len(teste)==len(p_lstm)
assert np.array_equal(teste.label.to_numpy(),y_lstm), 'Ordem das previsões não coincide com os IDs.'
if not MODO_RAPIDO:
    destino = REPO_DIR / 'resultados' / 'predictions_bilstm.csv'
    destino.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'id':teste.id,'y_true':y_lstm,'y_pred':p_lstm,
                  'score':np.round(scores_bilstm,6)}).to_csv(destino,index=False)
    print('Previsões oficiais da BiLSTM:',destino)
else:
    print('MODO_RAPIDO: nenhuma previsão final oficial foi exportada.')


---
## Modelo B · BERTimbau Base com fine-tuning
Usamos [neuralmind/bert-base-portuguese-cased](https://huggingface.co/neuralmind/bert-base-portuguese-cased) para classificação em duas classes. **Na primeira execução**, os pesos são baixados para o ambiente temporário do Colab; não é necessário baixar nada no seu PC. O cabeçalho de classificação é novo e será treinado, o que produz um aviso normal na carga do modelo.

**Importante:** se o Colab estiver sem GPU, ative uma GPU antes de executar esta etapa.

In [ ]:
assert torch.cuda.is_available(), 'Ative a GPU no menu Ambiente de execução > Alterar tipo de ambiente de execução.'
# Libere a GPU usada pelo primeiro modelo antes de carregar o BERT.
bilstm=bilstm.cpu()
del opt
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()

from datasets import Dataset as HFDataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)

NOME_BERT='neuralmind/bert-base-portuguese-cased'
MAX_TOKENS_BERT=256
tokenizer=AutoTokenizer.from_pretrained(NOME_BERT,do_lower_case=False)

def criar_dataset(tabela):
    dados=HFDataset.from_dict({'text':tabela.text.tolist(),'labels':tabela.label.astype(int).tolist()})
    return dados.map(lambda batch: tokenizer(batch['text'], truncation=True,max_length=MAX_TOKENS_BERT),
                     batched=True,remove_columns=['text'])

bert_treino=criar_dataset(treino)
bert_val=criar_dataset(validacao)
bert_teste=criar_dataset(teste)

modelo_bert=AutoModelForSequenceClassification.from_pretrained(
    NOME_BERT, num_labels=2,
    id2label={0:'VERDADEIRA',1:'FALSA'},
    label2id={'VERDADEIRA':0,'FALSA':1}
)
print('BERTimbau carregado.')

In [ ]:
def metricas_treino_bert(eval_pred):
    logits, y = eval_pred
    p=np.argmax(logits,axis=-1)
    return {'accuracy':accuracy_score(y,p),
            'f1_macro':f1_score(y,p,average='macro',zero_division=0)}

EPOCAS_BERT=1 if MODO_RAPIDO else 3
argumentos=TrainingArguments(
    output_dir=str(PASTA_RESULTADOS/'bert_checkpoints'),
    num_train_epochs=EPOCAS_BERT,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',   # não ocupar disco com checkpoints grandes
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED
)
trainer=Trainer(model=modelo_bert,args=argumentos,
                train_dataset=bert_treino,eval_dataset=bert_val,
                processing_class=tokenizer,
                data_collator=DataCollatorWithPadding(tokenizer),
                compute_metrics=metricas_treino_bert)
trainer.train()
print('Fim do treinamento do BERTimbau.')

In [ ]:
pred_bert=trainer.predict(bert_teste)
y_bert=np.asarray(pred_bert.label_ids)
p_bert=np.argmax(pred_bert.predictions,axis=1)
print('BERTimbau — TESTE:')
print(classification_report(y_bert,p_bert,target_names=['Verdadeira','Falsa'],digits=3,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_bert,p_bert,display_labels=['Verdadeira','Falsa'],cmap='Blues')
plt.title('BERTimbau — teste');plt.show()
metricas.append(calcular_metricas(y_bert,p_bert,'BERTimbau Base (fine-tuning)'))
previsoes['pred_bert']=p_bert

# Salve pesos só se precisar utilizar o modelo treinado depois: pode ocupar centenas de MB.
SALVAR_PESOS_BERT=False
if SALVAR_PESOS_BERT:
    destino=PASTA_RESULTADOS/'bertimbau_treinado'
    trainer.save_model(str(destino))
    tokenizer.save_pretrained(str(destino))
# Exportação padronizada — score = probabilidade de classe 1 (falsa).
assert np.array_equal(teste.label.to_numpy(),y_bert), 'Ordem do dataset BERT não coincide com o teste oficial.'
scores_bert = torch.softmax(torch.as_tensor(pred_bert.predictions),dim=1)[:,1].cpu().numpy()
if not MODO_RAPIDO:
    destino = REPO_DIR / 'resultados' / 'predictions_bertimbau.csv'
    destino.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'id':teste.id,'y_true':y_bert,'y_pred':p_bert,
                  'score':np.round(scores_bert,6)}).to_csv(destino,index=False)
    print('Previsões oficiais do BERTimbau:',destino)
else:
    print('MODO_RAPIDO: nenhuma previsão final oficial foi exportada.')


---
## 5. Comparação, erros e arquivos para o trabalho
Compare **F1-macro**, precisão e revocação da classe falsa, acurácia e matrizes de confusão. Analise casos discordantes. Não use o teste para escolher hiperparâmetros; para isso, use a validação.

> O Fake.Br é uma base rotulada com textos históricos. Classificar padrões aprendidos nela **não equivale a verificar a veracidade factual** de uma notícia nova.

In [ ]:
tabela_resultados=pd.DataFrame(metricas).sort_values('f1_macro',ascending=False)
display(tabela_resultados.style.format({c:'{:.3f}' for c in tabela_resultados.columns if c!='modelo'}))
tabela_resultados.to_csv(PASTA_RESULTADOS/'comparacao_deep_learning.csv',index=False)
previsoes['erro_bilstm']=previsoes.label!=previsoes.pred_bilstm
previsoes['erro_bert']=previsoes.label!=previsoes.pred_bert
previsoes.to_csv(PASTA_RESULTADOS/'previsoes_teste.csv',index=False)
print('Casos em que modelos discordaram:')
divergencias=previsoes[previsoes.pred_bilstm!=previsoes.pred_bert]
for _,r in divergencias.head(5).iterrows():
    print('\nRótulo real:',r.label,'| BiLSTM:',r.pred_bilstm,'| BERT:',r.pred_bert)
    print(r.text[:350].replace('\n',' '),'...')

print('\nArquivos gerados:')
for p in sorted(PASTA_RESULTADOS.iterdir()):
    if p.is_file(): print(p.name, f'({p.stat().st_size/1024:.0f} KiB)')

### 6. Arquivos para o grupo

No final de uma execução **com `MODO_RAPIDO=False`**, o notebook gera, em `/content/fakebr-classificacao/resultados/`:

- `predictions_bilstm.csv`
- `predictions_bertimbau.csv`

Cada CSV contém `id,y_true,y_pred,score`, exatamente como os resultados tradicionais. Baixe esses arquivos do Colab e envie para os colegas. Se não tiver permissão para escrever no repositório deles, solicite acesso ou abra um Pull Request por um fork. **Não suba os pesos do BERT.**

In [ ]:
import shutil
# Pacote contendo APENAS as previsões oficiais necessárias à comparação.
if MODO_RAPIDO:
    print('Teste rápido — desative MODO_RAPIDO e execute novamente para salvar previsões oficiais.')
else:
    arquivos=[REPO_DIR/'resultados'/'predictions_bilstm.csv',
              REPO_DIR/'resultados'/'predictions_bertimbau.csv']
    for p in arquivos:
        assert p.exists(), f'Arquivo esperado não foi criado: {p}'
    import zipfile
    zip_path=Path('/content/predicoes_deep_learning.zip')
    with zipfile.ZipFile(zip_path,'w') as z:
        for p in arquivos: z.write(p,arcname=p.name)
    print('ZIP de predições pronto:',zip_path)
    # Para baixar: descomente
    # from google.colab import files
    # files.download(str(zip_path))

### 7. O que apresentar

1. Os modelos usam a **mesma divisão oficial** de `data/split.csv` que o grupo tradicional.
2. A BiLSTM usa Word2Vec treinado apenas no conjunto de treino; o BERTimbau é ajustado por fine-tuning.
3. A escolha das épocas depende da validação; o teste não é usado para ajustes.
4. Resultados exportados em `id,y_true,y_pred,score`, comparáveis a SVM, Regressão Logística e Naive Bayes.
5. Analisar F1-macro, recall da classe falsa, matrizes de confusão e casos em que os modelos discordam.

**Cuidado:** isso é classificação com base em padrões de notícias antigas, e não checagem factual de notícias novas.